# Geopolitical Exposure Proof of Concept
## 02 — Point-in-Time Text Exposure Pipeline

This notebook builds the pre-event text corpus for the Russian-invasion pilot. It uses SEC EDGAR filings as the reproducible public source and accepts legally obtained earnings-call transcripts through the same document contract. It does not inspect event returns, select seeds, or call the OpenAI API unless the final execution switch is deliberately enabled.

## 1. Environment, paths, and frozen cutoff

The filing date is the public-availability date. Only documents filed on or before February 23, 2022 may enter the pilot. Raw documents and row-level outputs remain local and are excluded from Git.

In [ ]:
from pathlib import Path
from getpass import getpass
import hashlib, importlib, json, os, sys, time
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'geopolitical_exposure':
    GEO_ROOT, ROOT = ROOT, ROOT.parent
else:
    GEO_ROOT = ROOT / 'geopolitical_exposure'
if not GEO_ROOT.exists():
    raise FileNotFoundError('Open this notebook from Asset_Embeddings or geopolitical_exposure.')

sys.path.insert(0, str(GEO_ROOT / 'src'))
import document_contracts, edgar_pipeline
importlib.reload(document_contracts)
importlib.reload(edgar_pipeline)
from document_contracts import normalize_document, validate_documents
from edgar_pipeline import (download_company_tickers, filing_metadata_for_cik,
                            get_text, html_to_text, keyword_passages,
                            ticker_cik_crosswalk)

RAW = GEO_ROOT / 'documents' / 'raw'
PROCESSED = GEO_ROOT / 'documents' / 'processed'
RESULTS = GEO_ROOT / 'results'
MANIFESTS = GEO_ROOT / 'manifests'
for directory in [RAW, PROCESSED, RESULTS, MANIFESTS]:
    directory.mkdir(parents=True, exist_ok=True)

with open(GEO_ROOT / 'config' / 'russia_ukraine_pilot.json') as handle:
    EVENT = json.load(handle)
CUTOFF = pd.Timestamp(EVENT['information_cutoff'])
DOCUMENT_START = pd.Timestamp('2020-01-01')
FORMS = ('10-K', '10-Q', '8-K')
PILOT_FIRMS = 50
print('Project root:', ROOT)
print('Information cutoff:', CUTOFF.date())

## 2. Credentials and deliberate execution switches

SEC requests require an identifying user agent containing a real email address. This is not a secret. The OpenAI key is requested through a concealed field only when absent. Neither value is written into the notebook. Downloading and API use are separate switches.

In [ ]:
if not os.getenv('SEC_USER_AGENT'):
    os.environ['SEC_USER_AGENT'] = input(
        'SEC user agent (example: Academic research name@university.edu): '
    ).strip()
if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OpenAI API key (concealed): ')

DOWNLOAD_SEC_METADATA = False
DOWNLOAD_SEC_DOCUMENTS = False
RUN_OPENAI_EXTRACTION = False
RUN_FULL_PILOT = False
OPENAI_MODEL = 'gpt-5.6-luna'  # Frozen for the three-document extraction pilot

readiness = pd.DataFrame([{
    'sec_user_agent_ready': '@' in os.getenv('SEC_USER_AGENT', ''),
    'openai_key_ready': bool(os.getenv('OPENAI_API_KEY')),
    'openai_model_frozen': bool(OPENAI_MODEL),
    'download_sec_metadata': DOWNLOAD_SEC_METADATA,
    'download_sec_documents': DOWNLOAD_SEC_DOCUMENTS,
    'run_openai_extraction': RUN_OPENAI_EXTRACTION,
    'run_full_pilot': RUN_FULL_PILOT,
}])
display(readiness)

## 3. Freeze the 2021Q4 embedding universe and build the SEC bridge

The bridge uses exact CRSP-to-SEC ticker matches and explicitly flags ambiguities. This is a screening bridge, not yet a historically complete CIK–PERMCO crosswalk; unresolved firms are never silently assigned.

In [ ]:
embedding_path = (ROOT / 'active_passive_extension' / 'benchmarks' / 'all_s12_control' /
                  '2021q4' / 'rs_binary' / 'asset_embeddings_10d.parquet')
names_path = ROOT / 'data' / 'crosswalks' / 'crsp' / 'year=2021' / 'quarter=4' / 'stocknames_2021q4.parquet'
controls_path = ROOT / 'remaining_benchmarks' / 'return_comovement' / 'characteristics' / '2021q4.parquet'

embeddings = pd.read_parquet(embedding_path)
names = pd.read_parquet(names_path)
controls = pd.read_parquet(controls_path)
firm_column = 'permco' if 'permco' in embeddings.columns else 'firm_id'
universe = embeddings[[firm_column]].rename(columns={firm_column: 'permco'}).drop_duplicates()
universe['permco'] = pd.to_numeric(universe['permco'], errors='coerce').astype('Int64')
names = names[names['permco'].isin(universe['permco'])].copy()
names = names.sort_values(['permco', 'nameenddt']).drop_duplicates('permco', keep='last')

ticker_cache = RAW / 'sec_reference' / 'company_tickers_exchange.parquet'
if not ticker_cache.exists() and not DOWNLOAD_SEC_METADATA:
    raise FileNotFoundError(
        'SEC ticker cache is absent. Set DOWNLOAD_SEC_METADATA=True, rerun this cell, then turn it off.'
    )
sec_tickers = download_company_tickers(
    ticker_cache, os.environ['SEC_USER_AGENT'], refresh=False
)
bridge = ticker_cik_crosswalk(names, sec_tickers)
bridge.to_parquet(PROCESSED / 'permco_cik_exact_ticker_bridge.parquet', index=False)
bridge_audit = (bridge.groupby('cik_match_status', as_index=False)
                .agg(firms=('permco', 'nunique')))
bridge_audit['share'] = bridge_audit['firms'] / universe['permco'].nunique()
display(bridge_audit)

## 4. Select a bounded, size-stratified pilot

The first pass uses 50 uniquely linked firms spread across market-size quintiles. This prevents the API test from becoming only a mega-cap test while keeping EDGAR acquisition and manual validation manageable.

In [ ]:
eligible = (bridge.query("cik_match_status == 'unique_exact_ticker'")
            .drop_duplicates('permco')
            .merge(controls[['permco', 'log_market_equity']], on='permco', how='left'))
eligible['size_quintile'] = pd.qcut(eligible['log_market_equity'], 5, labels=False, duplicates='drop') + 1
per_group = PILOT_FIRMS // eligible['size_quintile'].nunique()
pilot = (eligible.sort_values(['size_quintile', 'log_market_equity'], ascending=[True, False])
         .groupby('size_quintile', group_keys=False).head(per_group)
         .head(PILOT_FIRMS).copy())
pilot['cik'] = pilot['cik'].astype(int)
pilot.to_parquet(PROCESSED / 'pilot_50_firms_frozen.parquet', index=False)
display(pilot.groupby('size_quintile').agg(firms=('permco', 'nunique'),
                                              median_log_me=('log_market_equity', 'median')).reset_index())
display(pilot[['permco', 'ticker', 'comnam', 'cik', 'sec_name', 'size_quintile']].head(15))

## 5. Retrieve and checkpoint filing metadata

This cell requests each pilot firm's public submissions JSON, then retains only 10-K, 10-Q, and 8-K filings from January 1, 2020 through the frozen cutoff. Raw JSON is cached by CIK, so reruns do not redownload completed work.

In [ ]:
metadata_path = PROCESSED / 'pilot_sec_filing_metadata_v3.parquet'
if metadata_path.exists():
    filing_metadata = pd.read_parquet(metadata_path)
    print('Loaded checkpointed filing metadata.')
elif DOWNLOAD_SEC_METADATA:
    frames, errors = [], []
    for row in pilot.itertuples(index=False):
        try:
            frame = filing_metadata_for_cik(
                row.cik, os.environ['SEC_USER_AGENT'], FORMS,
                str(DOCUMENT_START.date()), str(CUTOFF.date()), RAW / 'sec_submissions'
            )
            if not frame.empty:
                frame['permco'] = row.permco
                frame['crsp_ticker'] = row.ticker
                frames.append(frame)
        except Exception as error:
            errors.append({'cik': row.cik, 'permco': row.permco,
                           'error_type': type(error).__name__, 'error': str(error)})
    filing_metadata = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    filing_metadata.to_parquet(metadata_path, index=False)
    pd.DataFrame(errors).to_parquet(RESULTS / 'sec_metadata_errors_v3.parquet', index=False)
else:
    raise RuntimeError('Set DOWNLOAD_SEC_METADATA=True to build the filing metadata checkpoint.')

filing_audit = pd.DataFrame([{
    'pilot_firms': pilot['permco'].nunique(),
    'firms_with_filings': filing_metadata['permco'].nunique(),
    'qualifying_filings': len(filing_metadata),
    'latest_filing_date': filing_metadata['filingDate'].max(),
    'post_cutoff_filings': int((pd.to_datetime(filing_metadata['filingDate']) > CUTOFF).sum()),
}])
filing_audit.to_parquet(RESULTS / 'pilot_filing_metadata_audit_v3.parquet', index=False)
display(filing_audit)
display(filing_metadata.groupby('form').size().rename('filings').reset_index())

## 6. Freeze one primary pre-event document per firm

For the initial extraction-quality test, prefer the most recent 10-K; otherwise use the most recent 10-Q, then 8-K. The chosen accession and filing date are frozen before any event outcome is inspected.

In [ ]:
form_priority = {'10-K': 1, '10-Q': 2, '8-K': 3}
candidate_documents = filing_metadata.copy()
candidate_documents['form_priority'] = candidate_documents['form'].map(form_priority)
candidate_documents = candidate_documents.sort_values(
    ['permco', 'form_priority', 'filingDate'], ascending=[True, True, False]
)
selected_documents = candidate_documents.drop_duplicates('permco', keep='first').copy()
selected_documents['document_id'] = selected_documents['accessionNumber']
selected_documents.to_parquet(PROCESSED / 'pilot_documents_frozen.parquet', index=False)
display(selected_documents.groupby('form').size().rename('firms').reset_index())
assert pd.to_datetime(selected_documents['filingDate']).max() <= CUTOFF

## 7. Download and normalize selected EDGAR filings

Each selected filing is downloaded once, converted to readable text, hashed, and normalized to the shared document schema. Failures are logged without discarding successful files.

In [ ]:
normalized_path = PROCESSED / 'pilot_sec_documents.jsonl'
if normalized_path.exists():
    normalized_documents = [json.loads(line) for line in normalized_path.read_text().splitlines() if line.strip()]
    normalized_documents = validate_documents(normalized_documents)
    print('Loaded checkpointed normalized documents.')
elif DOWNLOAD_SEC_DOCUMENTS:
    normalized_documents, errors = [], []
    html_dir = RAW / 'sec_filings'
    html_dir.mkdir(parents=True, exist_ok=True)
    for row in selected_documents.itertuples(index=False):
        raw_path = html_dir / f'{row.accessionNumber}.html'
        try:
            html = raw_path.read_text(encoding='utf-8') if raw_path.exists() else get_text(
                row.source_uri, os.environ['SEC_USER_AGENT']
            )
            if not raw_path.exists():
                raw_path.write_text(html, encoding='utf-8')
            text = html_to_text(html)
            document_date = pd.to_datetime(row.reportDate, errors='coerce')
            if pd.isna(document_date):
                document_date = pd.Timestamp(row.filingDate)
            record = normalize_document({
                'document_id': row.accessionNumber, 'firm_id': str(int(row.permco)),
                'document_type': row.form,
                'document_date': document_date.date().isoformat(),
                'available_date': pd.Timestamp(row.filingDate).date().isoformat(),
                'source': 'SEC EDGAR', 'section': None, 'text': text,
                'source_uri': row.source_uri,
            })
            normalized_documents.append(record)
        except Exception as error:
            errors.append({'document_id': row.accessionNumber, 'permco': row.permco,
                           'error_type': type(error).__name__, 'error': str(error)})
    normalized_documents = validate_documents(normalized_documents)
    normalized_path.write_text(''.join(json.dumps(record) + '\n' for record in normalized_documents))
    pd.DataFrame(errors).to_parquet(RESULTS / 'sec_document_errors.parquet', index=False)
else:
    raise RuntimeError('Set DOWNLOAD_SEC_DOCUMENTS=True to build the local document checkpoint.')

document_audit = pd.DataFrame([{
    'documents': len(normalized_documents),
    'firms': len({record['firm_id'] for record in normalized_documents}),
    'latest_available_date': max(record['available_date'] for record in normalized_documents),
    'median_characters': float(np.median([len(record['text']) for record in normalized_documents])),
}])
document_audit.to_parquet(RESULTS / 'pilot_document_audit.parquet', index=False)
display(document_audit)

## 8. Optional transcript adapter

Legally obtained transcripts may be placed in `documents/raw/earnings_calls/` with `metadata.csv`. They remain a distinct document type, and the available date must be independently supplied. This cell validates them but never downloads transcripts from third-party sites.

In [ ]:
transcript_dir = RAW / 'earnings_calls'
transcript_metadata_path = transcript_dir / 'metadata.csv'
transcript_documents = []
if transcript_metadata_path.exists():
    transcript_metadata = pd.read_csv(transcript_metadata_path)
    required = {'document_id', 'firm_id', 'document_date', 'available_date',
                'source', 'source_uri', 'filename'}
    missing = required.difference(transcript_metadata.columns)
    if missing:
        raise ValueError(f'Transcript metadata is missing: {sorted(missing)}')
    for row in transcript_metadata.itertuples(index=False):
        transcript_documents.append(normalize_document({
            'document_id': row.document_id, 'firm_id': str(row.firm_id),
            'document_type': 'earnings_call', 'document_date': row.document_date,
            'available_date': row.available_date, 'source': row.source,
            'source_uri': row.source_uri, 'section': None,
            'text': (transcript_dir / row.filename).read_text(encoding='utf-8'),
        }))
    transcript_documents = validate_documents(transcript_documents)
print('Validated transcript documents:', len(transcript_documents))

## 9. Retrieval-first passage construction

The model will not receive entire filings. We first retrieve auditable windows around frozen geopolitical terms, substantially reducing cost and irrelevant context. Empty retrieval is retained as a meaningful no-evidence result.

In [ ]:
importlib.reload(edgar_pipeline)
keyword_passages = edgar_pipeline.keyword_passages
SEARCH_TERMS = [
    # Direct geographic and policy anchors
    'Russia', 'Russian', 'Ukraine', 'Ukrainian', 'Crimea', 'Belarus',
    'Black Sea', 'Eastern Europe', 'Commonwealth of Independent States',
    'OFAC', 'economic sanctions', 'trade sanctions', 'export controls',
    # Channel-specific business phrases; generic single words are excluded
    'oil and gas production', 'natural gas production', 'crude oil production',
    'oil and gas exploration', 'proved oil reserves', 'proved gas reserves',
    'Department of Defense', 'defense contractor', 'aerospace and defense',
    'military systems', 'weapons systems', 'missile systems',
    'fertilizer production', 'nitrogen fertilizer', 'potash production',
    'wheat production', 'grain exports', 'agricultural exports',
    'maritime shipping', 'ocean freight', 'shipping vessels', 'port operations',
]
all_documents = normalized_documents + transcript_documents
retrieval_records = []
for document in all_documents:
    passages = keyword_passages(document['text'], SEARCH_TERMS, radius=900, max_passages=30)
    retrieval_records.append({
        'document_id': document['document_id'], 'firm_id': document['firm_id'],
        'document_type': document['document_type'], 'available_date': document['available_date'],
        'content_sha256': document['content_sha256'], 'passage_count': len(passages),
        'retrieved_characters': sum(len(item['text']) for item in passages),
        'passages': passages,
    })
retrieval_path = PROCESSED / 'pilot_retrieval_records_v3.jsonl'
retrieval_path.write_text(''.join(json.dumps(record) + '\n' for record in retrieval_records))
retrieval_audit = pd.DataFrame([{key: value for key, value in record.items() if key != 'passages'}
                                for record in retrieval_records])
retrieval_audit.to_parquet(RESULTS / 'pilot_retrieval_audit_v3.parquet', index=False)
display(retrieval_audit[['passage_count', 'retrieved_characters']].describe())
print('Documents with at least one retrieved passage:', int(retrieval_audit.passage_count.gt(0).sum()))

## 10. Freeze the extraction prompt and schema

The extractor must cite supplied evidence spans, may not use outside knowledge, and must return no exposure when the passages do not support one. The prompt, schema, model ID, and source-document hash are recorded for every response.

In [ ]:
with open(GEO_ROOT / 'schemas' / 'exposure_output_schema.json') as handle:
    EXPOSURE_SCHEMA = json.load(handle)
CHANNELS = [item['channel'] for item in EVENT['channels']]
INSTRUCTIONS = '''You are a conservative research-data extractor. Use only the supplied
pre-event document passages. Do not use outside knowledge or infer exposure from the company
name. Return an exposure only when the quoted text directly supports the specified channel and
direction. A material disclosed business relationship to a channel—such as revenue, customers,
operations, assets, production, or facilities—counts as exposure even if the filing predates and
does not name the future event. Evidence must be an exact, shortest self-contained quote of no
more than 1000 characters from the supplied passage. If nothing is supported,
set no_supported_exposure=true and return an empty exposures array. A company name, patent
jurisdiction, generic compliance language, or generic industry membership without a disclosed
business relationship is not by itself
evidence of scenario exposure. evidence_start and evidence_end must be offsets in the original
document, computed from the supplied passage start plus the quote's location in that passage.'''
prompt_hash = hashlib.sha256(INSTRUCTIONS.encode()).hexdigest()
schema_hash = hashlib.sha256(json.dumps(EXPOSURE_SCHEMA, sort_keys=True).encode()).hexdigest()
prompt_manifest = {
    'event_id': EVENT['event_id'], 'information_cutoff': EVENT['information_cutoff'],
    'channels': CHANNELS, 'prompt_sha256': prompt_hash, 'schema_sha256': schema_hash,
    'model': OPENAI_MODEL or None, 'store': False, 'outcomes_inspected': False,
}
(MANIFESTS / 'text_extraction_design.json').write_text(json.dumps(prompt_manifest, indent=2))
display(pd.DataFrame([prompt_manifest]))

## 11. Cost-free API batch preview

This preview shows exactly which documents would be sent and how much retrieved text is involved. Review it before selecting and freezing an API model.

In [ ]:
api_queue = retrieval_audit.query('passage_count > 0').copy()
api_queue['approx_input_tokens'] = np.ceil(api_queue['retrieved_characters'] / 4).astype(int)
queue_summary = pd.DataFrame([{
    'queued_documents': len(api_queue),
    'queued_firms': api_queue['firm_id'].nunique(),
    'approx_input_tokens': int(api_queue['approx_input_tokens'].sum()),
    'api_enabled': RUN_OPENAI_EXTRACTION,
    'frozen_model': OPENAI_MODEL or None,
}])
queue_summary.to_parquet(RESULTS / 'api_queue_preview_v3.parquet', index=False)
display(queue_summary)
display(api_queue.sort_values('approx_input_tokens', ascending=False).head(10))

## 12. Guarded OpenAI structured extraction

This cell is intentionally blocked until the corpus audit is reviewed, a model ID is frozen, and `RUN_OPENAI_EXTRACTION=True` is set deliberately. Responses use strict JSON Schema and `store=False`. Start with at most three documents, manually verify every evidence quote, and only then scale.

In [ ]:
if not RUN_OPENAI_EXTRACTION:
    print('API extraction is disabled. Review the queue and freeze a model before enabling it.')
else:
    if not OPENAI_MODEL:
        raise ValueError('Set OPENAI_EXPOSURE_MODEL to the exact frozen model ID.')
    from openai import OpenAI
    client = OpenAI()
    retrieval_lookup = {record['document_id']: record for record in retrieval_records}
    output_dir = RESULTS / 'openai_responses_v2'
    output_dir.mkdir(parents=True, exist_ok=True)
    validation_document_ids = [
        '0001193125-22-046673',  # Landstar: direct Ukraine/Eastern Europe evidence
        '0001104659-21-094125',  # AAR: indirect defense-demand channel
        '0001564590-21-016719',  # Crinetics: patent-list false-positive challenge
    ]
    validation_queue = api_queue[api_queue['document_id'].isin(validation_document_ids)].copy()
    if set(validation_queue['document_id']) != set(validation_document_ids):
        raise ValueError('The frozen validation documents are not all present in the v3 API queue.')
    for row in validation_queue.itertuples(index=False):
        output_path = output_dir / f'{row.document_id}.json'
        if output_path.exists():
            continue
        record = retrieval_lookup[row.document_id]
        payload = {
            'firm_id': record['firm_id'], 'document_id': record['document_id'],
            'scenario_id': EVENT['event_id'], 'channel_definitions': EVENT['channels'],
            'passages': record['passages'],
        }
        response = client.responses.create(
            model=OPENAI_MODEL, instructions=INSTRUCTIONS,
            input=json.dumps(payload), store=False, reasoning={'effort': 'low'},
            text={'format': {'type': 'json_schema', 'name': 'scenario_exposure',
                             'strict': True, 'schema': EXPOSURE_SCHEMA}},
        )
        parsed = json.loads(response.output_text)
        audit_record = {
            'result': parsed, 'response_id': response.id, 'model': response.model,
            'usage': response.usage.model_dump() if response.usage else None,
            'prompt_sha256': prompt_hash, 'schema_sha256': schema_hash,
            'source_content_sha256': record['content_sha256'], 'store': False,
        }
        output_path.write_text(json.dumps(audit_record, indent=2))
    print('Completed guarded batch. Manually audit evidence before increasing the batch size.')

## 13. Guarded full-pilot extraction

The three challenge cases passed under prompt version 2. This cell extends the identical frozen model, prompt, schema, and retrieval rules to all 23 queued documents. Existing validation responses are reused, so enabling this stage calls only the remaining 20 documents. Every response is checkpointed immediately.

In [ ]:
full_output_dir = RESULTS / 'openai_responses_v2'
full_output_dir.mkdir(parents=True, exist_ok=True)
existing_ids = {path.stem for path in full_output_dir.glob('*.json')}
remaining_queue = api_queue[~api_queue['document_id'].isin(existing_ids)].copy()
full_pilot_status = pd.DataFrame([{
    'queued_documents': len(api_queue),
    'checkpointed_documents': len(existing_ids.intersection(set(api_queue.document_id))),
    'remaining_api_calls': len(remaining_queue),
    'full_pilot_enabled': RUN_FULL_PILOT,
    'model': OPENAI_MODEL,
    'prompt_sha256': prompt_hash,
    'schema_sha256': schema_hash,
}])
display(full_pilot_status)

if not RUN_FULL_PILOT:
    print('Full pilot is disabled. Set RUN_FULL_PILOT=True only after reviewing this status table.')
else:
    from openai import OpenAI
    client = OpenAI()
    retrieval_lookup = {record['document_id']: record for record in retrieval_records}
    failures = []
    for position, row in enumerate(remaining_queue.itertuples(index=False), start=1):
        output_path = full_output_dir / f'{row.document_id}.json'
        record = retrieval_lookup[row.document_id]
        payload = {
            'firm_id': record['firm_id'], 'document_id': record['document_id'],
            'scenario_id': EVENT['event_id'], 'channel_definitions': EVENT['channels'],
            'passages': record['passages'],
        }
        try:
            response = client.responses.create(
                model=OPENAI_MODEL, instructions=INSTRUCTIONS,
                input=json.dumps(payload), store=False, reasoning={'effort': 'low'},
                text={'format': {'type': 'json_schema', 'name': 'scenario_exposure',
                                 'strict': True, 'schema': EXPOSURE_SCHEMA}},
            )
            parsed = json.loads(response.output_text)
            audit_record = {
                'result': parsed, 'response_id': response.id, 'model': response.model,
                'usage': response.usage.model_dump() if response.usage else None,
                'prompt_sha256': prompt_hash, 'schema_sha256': schema_hash,
                'source_content_sha256': record['content_sha256'], 'store': False,
            }
            output_path.write_text(json.dumps(audit_record, indent=2))
            print(f'{position:02d}/{len(remaining_queue):02d} saved {row.document_id}')
        except Exception as error:
            failures.append({'document_id': row.document_id, 'firm_id': row.firm_id,
                             'error_type': type(error).__name__, 'error': str(error)})
            print(f'{position:02d}/{len(remaining_queue):02d} ERROR {row.document_id}: {type(error).__name__}')
    pd.DataFrame(failures).to_parquet(RESULTS / 'full_pilot_api_failures.parquet', index=False)
    completed_ids = {path.stem for path in full_output_dir.glob('*.json')}
    print('Completed queued documents:', len(completed_ids.intersection(set(api_queue.document_id))), '/', len(api_queue))

## 14. Deterministic evidence alignment and response audit

Model-produced offsets are never treated as authoritative. This cell preserves the raw offsets, finds every exact occurrence of each quotation in the original filing, chooses the unique match or the match nearest the raw offset, and records any unresolved ambiguity. It also verifies channel names, expected directions, logical consistency, hashes, and quote-length limits without overwriting raw API responses.

In [46]:
source_documents = {document['document_id']: document for document in normalized_documents + transcript_documents}
expected_directions = {item['channel']: item['expected_direction'] for item in EVENT['channels']}
aligned_records, audit_rows = [], []
response_paths = sorted((RESULTS / 'openai_responses_v2').glob('*.json'))

for response_path in response_paths:
    raw_record = json.loads(response_path.read_text())
    raw_result = raw_record['result']
    corrected_result = json.loads(json.dumps(raw_result))
    document = source_documents.get(raw_result['document_id'])
    if document is None:
        raise KeyError(f"Missing source document {raw_result['document_id']}")
    source_hash_matches = document['content_sha256'] == raw_record['source_content_sha256']
    logical_consistency = raw_result['no_supported_exposure'] == (len(raw_result['exposures']) == 0)
    if not raw_result['exposures']:
        audit_rows.append({
            'document_id': raw_result['document_id'], 'firm_id': raw_result['firm_id'],
            'channel': None, 'direction': None, 'score': None, 'confidence': None,
            'alignment_status': 'no_exposure', 'quote_occurrences': 0,
            'raw_evidence_start': None, 'raw_evidence_end': None,
            'corrected_evidence_start': None, 'corrected_evidence_end': None,
            'source_hash_matches': source_hash_matches,
            'logical_consistency': logical_consistency, 'channel_valid': True,
            'direction_valid': True, 'quote_within_limit': True,
        })
    for raw_exposure, corrected_exposure in zip(raw_result['exposures'], corrected_result['exposures']):
        quote = raw_exposure['evidence']
        positions, search_from = [], 0
        while quote:
            found = document['text'].find(quote, search_from)
            if found < 0:
                break
            positions.append(found)
            search_from = found + 1
        if len(positions) == 1:
            corrected_start, alignment_status = positions[0], 'unique_exact_quote'
        elif len(positions) > 1:
            corrected_start = min(positions, key=lambda value: abs(value - raw_exposure['evidence_start']))
            alignment_status = 'multiple_exact_quote_nearest_raw'
        else:
            corrected_start, alignment_status = None, 'quote_not_found'
        corrected_end = corrected_start + len(quote) if corrected_start is not None else None
        corrected_exposure['model_evidence_start'] = raw_exposure['evidence_start']
        corrected_exposure['model_evidence_end'] = raw_exposure['evidence_end']
        corrected_exposure['evidence_start'] = corrected_start
        corrected_exposure['evidence_end'] = corrected_end
        corrected_exposure['alignment_status'] = alignment_status
        channel = raw_exposure['channel']
        expected = expected_directions.get(channel)
        direction_valid = expected == 'ambiguous' or raw_exposure['direction'] == expected
        audit_rows.append({
            'document_id': raw_result['document_id'], 'firm_id': raw_result['firm_id'],
            'channel': channel, 'direction': raw_exposure['direction'],
            'score': raw_exposure['score'], 'confidence': raw_exposure['confidence'],
            'alignment_status': alignment_status, 'quote_occurrences': len(positions),
            'raw_evidence_start': raw_exposure['evidence_start'],
            'raw_evidence_end': raw_exposure['evidence_end'],
            'corrected_evidence_start': corrected_start, 'corrected_evidence_end': corrected_end,
            'source_hash_matches': source_hash_matches,
            'logical_consistency': logical_consistency,
            'channel_valid': channel in expected_directions, 'direction_valid': direction_valid,
            'quote_within_limit': len(quote) <= 1000,
        })
    aligned_records.append({
        'result': corrected_result, 'raw_response_file': response_path.name,
        'response_id': raw_record['response_id'], 'model': raw_record['model'],
        'usage': raw_record.get('usage'), 'prompt_sha256': raw_record['prompt_sha256'],
        'schema_sha256': raw_record['schema_sha256'],
        'source_content_sha256': raw_record['source_content_sha256'], 'store': raw_record['store'],
    })

aligned_path = PROCESSED / 'pilot_exposures_aligned_v2.jsonl'
aligned_path.write_text(''.join(json.dumps(record) + '\n' for record in aligned_records))
evidence_audit = pd.DataFrame(audit_rows)
evidence_audit.to_parquet(RESULTS / 'pilot_exposure_evidence_audit_v2.parquet', index=False)
audit_summary = pd.DataFrame([{
    'responses': len(aligned_records),
    'firms_with_supported_exposure': len({row['firm_id'] for row in audit_rows if row['channel'] is not None}),
    'exposures': sum(row['channel'] is not None for row in audit_rows),
    'unaligned_quotes': sum(row['alignment_status'] == 'quote_not_found' for row in audit_rows),
    'ambiguous_quotes': sum(row['alignment_status'] == 'multiple_exact_quote_nearest_raw' for row in audit_rows),
    'invalid_channels': sum(not row['channel_valid'] for row in audit_rows),
    'invalid_directions': sum(not row['direction_valid'] for row in audit_rows),
    'hash_failures': sum(not row['source_hash_matches'] for row in audit_rows),
    'logical_failures': sum(not row['logical_consistency'] for row in audit_rows),
    'overlong_quotes': sum(not row['quote_within_limit'] for row in audit_rows),
}])
audit_summary.to_parquet(RESULTS / 'pilot_exposure_audit_summary_v2.parquet', index=False)
display(audit_summary)
display(evidence_audit)
print('Aligned results:', aligned_path)

,responses,firms_with_supported_exposure,exposures,unaligned_quotes,ambiguous_quotes,invalid_channels,invalid_directions,hash_failures,logical_failures,overlong_quotes
0,23,14,15,1,1,0,1,0,0,0


,document_id,firm_id,channel,direction,score,confidence,alignment_status,quote_occurrences,raw_evidence_start,raw_evidence_end,corrected_evidence_start,corrected_evidence_end,source_hash_matches,logical_consistency,channel_valid,direction_valid,quote_within_limit
0,0000019617-22-000272,20436,None,None,NaN,NaN,no_exposure,0,NaN,NaN,NaN,NaN,True,True,True,True,True
1,0000049196-22-000023,2093,None,None,NaN,NaN,no_exposure,0,NaN,NaN,NaN,NaN,True,True,True,True,True
2,0000102037-21-000012,21838,russia_ukraine_operations_revenue,negative,1.00,0.98,unique_exact_quote,1,49045.0,49365.0,49263.0,49597.0,True,True,True,True,True
3,0000105634-21-000044,14268,russia_ukraine_operations_revenue,negative,0.90,0.96,unique_exact_quote,1,106647.0,106934.0,106726.0,107013.0,True,True,True,True,True
4,0000320193-21-000105,7,russia_ukraine_operations_revenue,negative,-0.70,0.91,unique_exact_quote,1,225241.0,225307.0,225499.0,225568.0,True,True,True,True,True
5,0000705432-21-000013,16107,None,None,NaN,NaN,no_exposure,0,NaN,NaN,NaN,NaN,True,True,True,True,True
6,0000802481-22-000012,53305,None,None,NaN,NaN,no_exposure,0,NaN,NaN,NaN,NaN,True,True,True,True,True
7,0000864749-22-000044,10691,None,None,NaN,NaN,no_exposure,0,NaN,NaN,NaN,NaN,True,True,True,True,True
8,0001045810-21-000010,16382,shipping_logistics,ambiguous,0.35,0.93,unique_exact_quote,1,102652.0,102775.0,102653.0,102776.0,True,True,True,False,True
9,0001104659-21-094125,20000,defense_demand,positive,0.85,0.98,unique_exact_quote,1,268819.0,269039.0,268951.0,269163.0,True,True,True,True,True


Aligned results: /Users/harmandeep/Desktop/Projects/Asset_Embeddings/geopolitical_exposure/documents/processed/pilot_exposures_aligned_v2.jsonl


## 15. Manual adjudication and high-precision seed freeze

Every supported model classification receives a documented researcher decision before outcomes are inspected. Accepted seeds require a disclosed business relationship to the frozen channel. Generic compliance language is rejected, while plausible but indirect evidence is retained as borderline rather than silently discarded. Raw model scores and offsets remain unchanged in the API logs; this table adds canonical evidence and a consistently signed score.

In [47]:
decision_map = {
    ('21838', 'russia_ukraine_operations_revenue'): ('accepted_seed', 'Direct leaf-tobacco business operations in Russia.'),
    ('14268', 'russia_ukraine_operations_revenue'): ('borderline', 'Russia-related oil-price sensitivity is relevant but not direct Russian operations.'),
    ('7', 'russia_ukraine_operations_revenue'): ('accepted_seed', 'Products were imported into Russia to conduct business there.'),
    ('16382', 'shipping_logistics'): ('rejected', 'Asian distribution locations do not establish Russia–Ukraine shipping exposure.'),
    ('20000', 'defense_demand'): ('accepted_seed', 'Defense and related government customers represent a material share of sales.'),
    ('37682', 'russia_ukraine_operations_revenue'): ('accepted_seed', 'Manufacturing or assembly facilities were located in Russia.'),
    ('51198', 'russia_ukraine_operations_revenue'): ('accepted_seed', 'The filing identifies an office in Russia; evidence text is manually corrected below.'),
    ('28858', 'russia_ukraine_operations_revenue'): ('accepted_seed', 'Operations in Ukraine are explicitly exposed to escalation and armed conflict.'),
    ('53339', 'russia_ukraine_operations_revenue'): ('accepted_seed', 'The filing identifies operations or sales offices in Russia.'),
    ('53896', 'russia_ukraine_operations_revenue'): ('accepted_seed', 'The Russian ruble materially affected reported revenue translation.'),
    ('55370', 'sanctions_sensitive_supply_chain'): ('rejected', 'Generic sanctions compliance language does not identify Russia-specific business exposure.'),
    ('8048', 'sanctions_sensitive_supply_chain'): ('rejected', 'General import-control language is insufficient for a high-precision direct seed.'),
    ('21121', 'sanctions_sensitive_supply_chain'): ('rejected', 'Generic international sourcing and compliance language is not scenario-specific.'),
    ('20253', 'russia_ukraine_operations_revenue'): ('accepted_seed', 'The filing identifies operations and customer operations in Russia.'),
    ('20253', 'oil_gas_production'): ('accepted_seed', 'Products and services directly support upstream oil and gas production.'),
}

commvault_quote = (
    'In addition, we have offices in the United States in California and Texas; and outside the United States in '
    'Australia, Austria, Belgium, Brazil, Canada, China, Denmark, France, Germany, Hong Kong, India, Ireland, Israel, '
    'Italy, Japan, Netherlands, Poland, Russia, Saudi Arabia, Singapore, South Africa, South Korea, Spain, Sweden, '
    'Switzerland, Taiwan, United Arab Emirates, and United Kingdom.'
)
manual_rows = []
for aligned_record in aligned_records:
    result = aligned_record['result']
    document = source_documents[result['document_id']]
    for exposure in result['exposures']:
        key = (str(result['firm_id']), exposure['channel'])
        if key not in decision_map:
            raise KeyError(f'Missing manual decision for {key}')
        decision, rationale = decision_map[key]
        canonical_quote = exposure['evidence']
        canonical_start = exposure['evidence_start']
        alignment_status = exposure['alignment_status']
        manual_override = False
        if key == ('51198', 'russia_ukraine_operations_revenue'):
            canonical_quote = commvault_quote
            canonical_start = document['text'].find(canonical_quote)
            if canonical_start < 0:
                raise ValueError('Verified Commvault evidence could not be located in the source filing.')
            alignment_status = 'manual_exact_quote_correction'
            manual_override = True
        canonical_end = canonical_start + len(canonical_quote) if canonical_start is not None else None
        direction_sign = {'negative': -1.0, 'positive': 1.0, 'ambiguous': 0.0, 'none': 0.0}[exposure['direction']]
        signed_score = direction_sign * abs(float(exposure['score']))
        manual_rows.append({
            'document_id': result['document_id'], 'firm_id': str(result['firm_id']),
            'channel': exposure['channel'], 'direction': exposure['direction'],
            'raw_model_score': exposure['score'], 'signed_score': signed_score,
            'confidence': exposure['confidence'], 'decision': decision,
            'decision_rationale': rationale, 'canonical_evidence': canonical_quote,
            'canonical_evidence_start': canonical_start, 'canonical_evidence_end': canonical_end,
            'alignment_status': alignment_status, 'manual_evidence_override': manual_override,
            'source_content_sha256': document['content_sha256'],
            'prompt_sha256': aligned_record['prompt_sha256'],
            'schema_sha256': aligned_record['schema_sha256'], 'model': aligned_record['model'],
        })

manual_adjudication = pd.DataFrame(manual_rows)
assert len(manual_adjudication) == len(decision_map) == 15
assert manual_adjudication.apply(
    lambda row: source_documents[row.document_id]['text'][int(row.canonical_evidence_start):int(row.canonical_evidence_end)]
                == row.canonical_evidence, axis=1
).all()
manual_adjudication.to_parquet(RESULTS / 'manual_exposure_adjudication_v1.parquet', index=False)

seed_portfolio = manual_adjudication.query("decision == 'accepted_seed'").copy()
seed_portfolio['firm_id'] = pd.to_numeric(seed_portfolio['firm_id']).astype('Int64')
seed_portfolio = seed_portfolio.merge(
    pilot[['permco', 'ticker', 'comnam']].drop_duplicates('permco'),
    left_on='firm_id', right_on='permco', how='left', validate='many_to_one'
)
seed_portfolio.to_parquet(PROCESSED / 'text_exposure_seeds_frozen_v1.parquet', index=False)
seed_hash_columns = ['firm_id', 'channel', 'direction', 'signed_score', 'confidence',
                     'canonical_evidence', 'source_content_sha256']
seed_payload = seed_portfolio[seed_hash_columns].sort_values(['firm_id', 'channel']).to_dict('records')
seed_sha256 = hashlib.sha256(json.dumps(seed_payload, sort_keys=True, default=str).encode()).hexdigest()
seed_manifest = {
    'event_id': EVENT['event_id'], 'information_cutoff': EVENT['information_cutoff'],
    'outcomes_inspected': False, 'accepted_firms': int(seed_portfolio.firm_id.nunique()),
    'accepted_firm_channel_seeds': int(len(seed_portfolio)),
    'borderline_exposures': int((manual_adjudication.decision == 'borderline').sum()),
    'rejected_exposures': int((manual_adjudication.decision == 'rejected').sum()),
    'seed_sha256': seed_sha256, 'model': OPENAI_MODEL,
    'prompt_sha256': prompt_hash, 'schema_sha256': schema_hash,
}
(MANIFESTS / 'text_exposure_seed_freeze_v1.json').write_text(json.dumps(seed_manifest, indent=2))
display(manual_adjudication.groupby('decision').size().rename('exposures').reset_index())
display(seed_portfolio[['firm_id', 'ticker', 'comnam', 'channel', 'direction',
                        'signed_score', 'confidence']].sort_values(['direction', 'firm_id']))
print('Frozen seed manifest:', seed_manifest)

,decision,exposures
0,accepted_seed,10
1,borderline,1
2,rejected,4


,firm_id,ticker,comnam,channel,direction,signed_score,confidence
1,7,AAPL,APPLE INC,russia_ukraine_operations_revenue,negative,-0.70,0.91
8,20253,BKR,BAKER HUGHES CO,russia_ukraine_operations_revenue,negative,-0.90,0.96
0,21838,UVV,UNIVERSAL CORP,russia_ukraine_operations_revenue,negative,-1.00,0.98
5,28858,LSTR,LANDSTAR SYSTEM INC,russia_ukraine_operations_revenue,negative,-0.95,0.99
3,37682,VC,VISTEON CORP,russia_ukraine_operations_revenue,negative,-0.85,0.98
4,51198,CVLT,COMMVAULT SYSTEMS INC,russia_ukraine_operations_revenue,negative,-0.45,0.86
6,53339,GNRC,GENERAC HOLDINGS INC,russia_ukraine_operations_revenue,negative,-0.80,0.98
7,53896,XYL,XYLEM INC,russia_ukraine_operations_revenue,negative,-0.55,0.84
2,20000,AIR,A A R CORP,defense_demand,positive,0.85,0.98
9,20253,BKR,BAKER HUGHES CO,oil_gas_production,positive,0.75,0.96


Frozen seed manifest: {'event_id': 'russia_ukraine_invasion_2022', 'information_cutoff': '2022-02-23', 'outcomes_inspected': False, 'accepted_firms': 9, 'accepted_firm_channel_seeds': 10, 'borderline_exposures': 1, 'rejected_exposures': 4, 'seed_sha256': 'f159fbc59073a27619fd46c34018c440c2f506ec5be10fb086801eaeb0dc8efa', 'model': 'gpt-5.6-luna', 'prompt_sha256': 'c70a941cfd181ae885e3df088350f1062cc9e1d9d3b9f1e16570489b7efffe6b', 'schema_sha256': '14ce503a5718f28c118a6add7ff99e6073589e0cf3af2ca99cca0aac2b5316f6'}


## 16. Decision gate

The next stage is embedding expansion, not event evaluation. Proceed only if identifier coverage, document coverage, cutoff integrity, API response coverage, canonical evidence integrity, and signed seed availability pass. A low retrieval-hit rate is not treated as failure because most firms should have no narrow scenario evidence. Outcomes remain quarantined.

In [48]:
decision_gate = pd.DataFrame([
    {'gate': 'unique_cik_bridge', 'value': float((bridge.cik_match_status == 'unique_exact_ticker').mean()),
     'passed': float((bridge.cik_match_status == 'unique_exact_ticker').mean()) >= 0.70},
    {'gate': 'pilot_document_coverage', 'value': len(normalized_documents) / len(pilot),
     'passed': len(normalized_documents) / len(pilot) >= 0.80},
    {'gate': 'cutoff_integrity',
     'value': float(all(pd.Timestamp(record['available_date']) <= CUTOFF for record in normalized_documents)),
     'passed': all(pd.Timestamp(record['available_date']) <= CUTOFF for record in normalized_documents)},
    {'gate': 'api_response_coverage', 'value': len(aligned_records) / len(api_queue),
     'passed': len(aligned_records) == len(api_queue)},
    {'gate': 'canonical_evidence_integrity', 'value': float(manual_adjudication.apply(
         lambda row: source_documents[row.document_id]['text'][int(row.canonical_evidence_start):int(row.canonical_evidence_end)]
                     == row.canonical_evidence, axis=1).mean()),
     'passed': bool(manual_adjudication.apply(
         lambda row: source_documents[row.document_id]['text'][int(row.canonical_evidence_start):int(row.canonical_evidence_end)]
                     == row.canonical_evidence, axis=1).all())},
    {'gate': 'manual_positive_acceptance_rate',
     'value': float((manual_adjudication.decision == 'accepted_seed').mean()),
     'passed': float((manual_adjudication.decision == 'accepted_seed').mean()) >= 0.50},
    {'gate': 'both_signed_seed_directions',
     'value': float(seed_portfolio.direction.nunique()),
     'passed': {'negative', 'positive'}.issubset(set(seed_portfolio.direction))},
    {'gate': 'outcome_quarantine', 'value': 1.0, 'passed': True},
])
decision_gate.to_parquet(RESULTS / 'text_pipeline_decision_gate_v2.parquet', index=False)
display(decision_gate)
print('Ready for embedding expansion:', bool(decision_gate.passed.all()))

,gate,value,passed
0,unique_cik_bridge,0.791136,True
1,pilot_document_coverage,0.980000,True
2,cutoff_integrity,1.000000,True
3,api_response_coverage,1.000000,True
4,canonical_evidence_integrity,1.000000,True
5,manual_positive_acceptance_rate,0.666667,True
6,both_signed_seed_directions,2.000000,True
7,outcome_quarantine,1.000000,True


Ready for embedding expansion: True
